# Experimentação

Este notebook orquestra a Fase 1 e 2 da etapa de experimentação, seguindo o protocolo descrito em `docs/experimentation.md`.

## Objetivos

## Setups e Imports

In [1]:
import sys
from pathlib import Path

# Garante que a raiz do projeto esta no sys.path
ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate project root.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Manipulacao de Dados
import pandas as pd
import numpy as np

# Visualizacao de Dados
import matplotlib.pyplot as plt
import seaborn as sns

# Pre-processamento e Modelagem
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV,
)
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Metricas de Avaliacao
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    make_scorer,
)

# Modelos de Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from src.utils.exp import (
    MLPClassifierWrapper,
    build_k_grid,
    evaluate_round3_model_strategies,
    extract_selected_feature_names,
    format_selected_features_log,
    get_processed_feature_names,
    summarize_grid_search_results,
)

# importando os transformers customizados
from src.features.geo_transformer import GeoTransformer
from src.features.feature_engineer_transformer import FeatureEngineerTransformer
from src.utils.logging_config import get_logger

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# MLflow
# ATENCAO: o servidor do MLflow precisa estar rodando antes de executar este notebook.
# Em um terminal separado, execute:
#   mlflow server --host 127.0.0.1 --port 5000
# Sem isso, as celulas de tracking falharao com ConnectionRefusedError.
import hashlib
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://localhost:5000")

## Processamento dos Dados

In [2]:
# dataload
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [3]:
# Informações gerais sobre o dataset
print("=== INFORMAÇÕES GERAIS DO DATASET ===\n")
print(df.info())

# Shape
print("\n=== SHAPE DO DATASET ===")
print(f"Linhas: {df.shape[0]}, Colunas: {df.shape[1]}")

# Colunas
print("\n=== COLUNAS DO DATASET ===")
print(df.columns.tolist())

=== INFORMAÇÕES GERAIS DO DATASET ===

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null 

In [4]:
# Transformação da coluna 'Total Charges' para numérica, tratando erros e preenchendo valores ausentes com 0.
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'] = df['Total Charges'].fillna(0)

In [5]:
# dropando colunas irrelevantes para a modelagem
drop_cols = [
    'Country',
    'State',
    'Lat Long',
    'Churn Label',
    'Churn Reason',
    'Count'
]


df.drop(columns=drop_cols, inplace=True)

In [6]:
target = "Churn Value"
meta_cols = ["CLTV", "CustomerID"]

feature_cols = [
    col for col in df.columns
    if col not in [target] + meta_cols
]

X = df[feature_cols]
y = df[target]

metadata = df[meta_cols]


## Splits e Validação

In [7]:
# Protocolo de validação cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# 
X_train_val, X_test, y_train_val, y_test, metadata_train_val, metadata_test = train_test_split(
    X,
    y,
    metadata,
    test_size=0.3,
    stratify=y,
    random_state=42
)

# Checando os Splits
print("=== SPLITS ===")
print(f"Treino/Validação: {X_train_val.shape[0]} amostras")
print(f"Teste: {X_test.shape[0]} amostras")

# Checando a distribuição da variável alvo nos splits
print("\n=== DISTRIBUIÇÃO DA VARIÁVEL ALVO NOS SPLITS ===")
print("Treino/Validação:")
print(y_train_val.value_counts(normalize=True))
print("\nTeste:")
print(y_test.value_counts(normalize=True))

# Checando o formato dos dados
print("\n=== FORMATO DOS DADOS ===")
print(f"X_train_val: {X_train_val.shape}")
print(f"y_train_val: {y_train_val.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")


=== SPLITS ===
Treino/Validação: 4930 amostras
Teste: 2113 amostras

=== DISTRIBUIÇÃO DA VARIÁVEL ALVO NOS SPLITS ===
Treino/Validação:
Churn Value
0    0.734686
1    0.265314
Name: proportion, dtype: float64

Teste:
Churn Value
0    0.734501
1    0.265499
Name: proportion, dtype: float64

=== FORMATO DOS DADOS ===
X_train_val: (4930, 24)
y_train_val: (4930,)
X_test: (2113, 24)
y_test: (2113,)


## Baseline Inicial

- Treina Dummy e Regressão Logística e compara com a MLP e outros modelos de Árvores

In [8]:
# Definindo a etapa de pré-processamento para variáveis categóricas com OHE e numéricas com passthrough
ohe = OneHotEncoder(handle_unknown="ignore")
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ohe, make_column_selector(dtype_include=["object", "category"])),
        ("num", "passthrough", make_column_selector(dtype_exclude=["object", "category"])),
    ],
    remainder="drop",
)

# Dicionário de Pipelines para cada modelo
baseline_params = dict(
    drop_churn_score=True,
    add_engagement_score=False,
    add_tenure_group=False,
    add_tenure_log=False,
    add_contract_ordinal=False,
    add_family_stability=False,
    add_fiber_no_support=False,
    add_support_gap_count=False,
    add_payment_automatic_flag=False,
    add_electronic_check_flag=False,
    add_paperless_echeck_flag=False,
    add_price_pressure_ratio=False,
)

models = {
    "Dummy": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]),
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "DecisionTree": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", DecisionTreeClassifier(
            random_state=42,
            class_weight="balanced",
        )),
    ]),
    "RandomForest": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            random_state=42,
            n_jobs=-1,
            class_weight="balanced_subsample",
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}


In [9]:
# definindo o scoring para avaliação dos modelos
scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": make_scorer(recall_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}

In [20]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")
    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{m: cv_res[f"test_{m}"] for m in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{m}_mean": cv_res[f"test_{m}"].mean() for m in metrics},
        "pr_auc_std": cv_res["test_pr_auc"].std(),
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,pr_auc_std,fit_time_mean_s,score_time_mean_s
0,XGBoost,0.9512,0.9802,0.8738,0.8404,0.8566,0.0112,0.0719,0.0272
1,LogisticRegression,0.9393,0.9759,0.9297,0.7827,0.8497,0.0105,0.0418,0.0192
2,MLP,0.9357,0.9744,0.9296,0.7707,0.8422,0.0111,1.0929,0.0234


### Validando o Wrapper

- Aplicação da MLP fora do pipeline para validação dos resultados do Wrapper.

In [11]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
)

from src.models.mlp import MLP, evaluate, train_with_early_stopping

# ---------- Config ----------
MLP_EPOCHS = 80
MLP_BATCH_SIZE = 64
MLP_LR = 1e-3
MLP_WD = 1e-5
MLP_HIDDEN_DIM = 64
MLP_DROPOUT = 0.0
MLP_THRESHOLD = 0.5

ES_PATIENCE = int(np.ceil(0.2 * MLP_EPOCHS))
ES_MIN_DELTA = 1e-3
ES_VAL_SIZE = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _rows(X, idx):
    return X.iloc[idx] if hasattr(X, "iloc") else X[idx]


def _to_dense_float32(x):
    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x, dtype=np.float32)


baseline_fe = FeatureEngineerTransformer(**baseline_params)
baseline_geo = GeoTransformer(strategy="drop")

mlp_manual_folds = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train_val, y_train_val), start=1):
    fit_start = time.perf_counter()

    X_tr_raw = _rows(X_train_val, tr_idx)
    X_va_raw = _rows(X_train_val, va_idx)
    y_tr = np.asarray(_rows(y_train_val, tr_idx), dtype=np.float32)
    y_va = np.asarray(_rows(y_train_val, va_idx), dtype=np.float32)

    X_tr_base = baseline_fe.fit_transform(X_tr_raw, y_tr)
    X_tr_base = baseline_geo.fit_transform(X_tr_base, y_tr)
    X_va_base = baseline_fe.transform(X_va_raw)
    X_va_base = baseline_geo.transform(X_va_base)

    prep_fold = clone(preprocessor)
    X_tr_enc = prep_fold.fit_transform(X_tr_base, y_tr)
    X_va_enc = prep_fold.transform(X_va_base)

    idx_all = np.arange(len(y_tr))
    idx_tr, idx_es = train_test_split(
        idx_all,
        test_size=ES_VAL_SIZE,
        stratify=y_tr,
        random_state=42 + fold,
    )

    X_tr_fit = X_tr_enc[idx_tr]
    y_tr_fit = y_tr[idx_tr]
    X_tr_es = X_tr_enc[idx_es]
    y_tr_es = y_tr[idx_es]

    scaler = StandardScaler(with_mean=False)
    X_tr_fit = scaler.fit_transform(X_tr_fit)
    X_tr_es = scaler.transform(X_tr_es)
    X_va_sc = scaler.transform(X_va_enc)

    X_tr_fit = _to_dense_float32(X_tr_fit)
    X_tr_es = _to_dense_float32(X_tr_es)
    X_va_sc = _to_dense_float32(X_va_sc)

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.tensor(X_tr_fit, dtype=torch.float32),
            torch.tensor(y_tr_fit, dtype=torch.float32),
        ),
        batch_size=MLP_BATCH_SIZE,
        shuffle=True,
    )
    es_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.tensor(X_tr_es, dtype=torch.float32),
            torch.tensor(y_tr_es, dtype=torch.float32),
        ),
        batch_size=MLP_BATCH_SIZE,
        shuffle=False,
    )

    model = MLP(
        input_dim=X_tr_fit.shape[1],
        hidden_dim=MLP_HIDDEN_DIM,
        output_dim=1,
        dropout=MLP_DROPOUT,
    ).to(DEVICE)

    pos = float((y_tr_fit == 1).sum())
    neg = float((y_tr_fit == 0).sum())
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=MLP_LR, weight_decay=MLP_WD)

    epochs_trained = train_with_early_stopping(
        model,
        train_loader,
        es_loader,
        optimizer,
        criterion,
        device=DEVICE,
        max_epochs=MLP_EPOCHS,
        patience=ES_PATIENCE,
        min_delta=ES_MIN_DELTA,
        threshold=MLP_THRESHOLD,
    )
    best_es_loss, _ = evaluate(
        model,
        es_loader,
        criterion,
        device=DEVICE,
        threshold=MLP_THRESHOLD,
    )
    fit_time_s = time.perf_counter() - fit_start

    score_start = time.perf_counter()
    X_va_t = torch.tensor(X_va_sc, dtype=torch.float32).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(X_va_t).squeeze(1)
        prob = torch.sigmoid(logits).cpu().numpy()

    pred = (prob >= MLP_THRESHOLD).astype(int)

    mlp_manual_folds.append(
        {
            "fold": fold,
            "epochs_trained": epochs_trained,
            "best_es_loss": best_es_loss,
            "pr_auc": average_precision_score(y_va, prob),
            "roc_auc": roc_auc_score(y_va, prob),
            "recall": recall_score(y_va, pred, zero_division=0),
            "precision": precision_score(y_va, pred, zero_division=0),
            "f1": f1_score(y_va, pred, zero_division=0),
            "fit_time_s": fit_time_s,
            "score_time_s": time.perf_counter() - score_start,
        }
    )

mlp_fold_results = pd.DataFrame(mlp_manual_folds)

mlp_cv_summary = pd.DataFrame(
    [
        {
            "model": "MLP_manual",
            "pr_auc_mean": mlp_fold_results["pr_auc"].mean(),
            "pr_auc_std": mlp_fold_results["pr_auc"].std(),
            "roc_auc_mean": mlp_fold_results["roc_auc"].mean(),
            "recall_mean": mlp_fold_results["recall"].mean(),
            "precision_mean": mlp_fold_results["precision"].mean(),
            "f1_mean": mlp_fold_results["f1"].mean(),
            "fit_time_mean_s": mlp_fold_results["fit_time_s"].mean(),
            "score_time_mean_s": mlp_fold_results["score_time_s"].mean(),
        }
    ]
)

display(mlp_cv_summary.round(4))

,model,pr_auc_mean,pr_auc_std,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,MLP_manual,0.6642,0.0331,0.8542,0.8073,0.5231,0.6344,0.9296,0.0036


### Conclusão

Na tabela de baselines, a `LogisticRegression` apresentou o melhor desempenho geral, com `PR-AUC = 0.6782` e `ROC-AUC = 0.8576`, ficando levemente acima da `MLP` (`PR-AUC = 0.6728` e `ROC-AUC = 0.8551`). Ainda assim, a diferença entre os dois modelos foi pequena, e a `MLP` superou o benchmark de Árvore selecionado nesta rodada, o `XGBoost` (`PR-AUC = 0.6492`). Isso indica que, já na base original, a MLP se mostrou competitiva em relação ao baseline linear e ao benchmark não linear.

Na etapa de validação do Wrapper do MLP, a `MLP` dentro do `Pipeline` manteve desempenho consistente e até ligeiramente superior à versão manual fora do pipeline. Com isso, o wrapper foi validado com sucesso para uso no fluxo de experimentação, trazendo a vantagem de encapsular o preprocessamento e a validação cruzada dentro da mesma estrutura, com menor risco de leakage e maior facilidade para evoluir o pipeline com feature engineering e seleção de features.

### Logging no MLflow

## Feature Engineering

**Objetivo**: 
- Adicionar poder preditivo aos modelos de forma controlada

### Round 1 - FE orientada a hipótese

- adicionando features a partir de hipóteses construídas a partir da EDA;

In [12]:
round1_fe_params = dict(
    drop_churn_score=True,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)


models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round1_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round1_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round1_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}

In [13]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{metric}_mean": cv_res[f"test_{metric}"].mean() for metric in metrics},
        "pr_auc_std": cv_res["test_pr_auc"].std(),
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,pr_auc_std,fit_time_mean_s,score_time_mean_s
0,LogisticRegression,0.6893,0.8625,0.8234,0.5382,0.6508,0.0158,0.0576,0.0208
1,MLP,0.6874,0.8600,0.8341,0.5195,0.6399,0.0106,0.7594,0.0239
2,XGBoost,0.6465,0.8418,0.6636,0.5752,0.6160,0.0180,0.0676,0.0272


### Round 2 - Adicionando `Churn Score`

- Verificando o ganho com stacking de outro modelo;

In [14]:
round2_fe_params = dict(
    drop_churn_score=False,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)


models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round2_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round2_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round2_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}

In [15]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{metric}_mean": cv_res[f"test_{metric}"].mean() for metric in metrics},
        "pr_auc_std": cv_res["test_pr_auc"].std(),
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,pr_auc_std,fit_time_mean_s,score_time_mean_s
0,XGBoost,0.9512,0.9802,0.8738,0.8404,0.8566,0.0112,0.0674,0.0269
1,LogisticRegression,0.9393,0.9759,0.9297,0.7827,0.8497,0.0105,0.0462,0.0208
2,MLP,0.9358,0.9744,0.9312,0.7682,0.8413,0.0101,1.1108,0.0236


### Round 3 - Testando encoding para City

Definindo as mesmas features para todos as estratégias de encoding

In [16]:
round3_fe_params = dict(
    drop_churn_score=False,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)


Configuração das estratégias por modelo

In [22]:
assert "City" in X_train_val.columns, "City precisa estar presente em X_train_val para o Round 3."

round3_metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]

round3_strategy_specs = {
    "LogisticRegression": [
        ("frequency", "LogisticRegression_Frequency"),
        ("target", "LogisticRegression_Target"),
        ("geo_cluster", "LogisticRegression_Geo_Cluster"),
        ("zip_region", "LogisticRegression_ZIP"),
        ("risk_band", "LogisticRegression_Risk_Band"),
    ],
    "XGBoost": [
        ("frequency", "XGBoost_Frequency"),
        ("target", "XGBoost_Target"),
        ("geo_cluster", "XGBoost_Geo_Cluster"),
        ("zip_region", "XGBoost_ZIP"),
        ("risk_band", "XGBoost_Risk_Band"),
    ],
    "MLP": [
        ("frequency", "MLP_Frequency"),
        ("target", "MLP_Target"),
        ("geo_cluster", "MLP_Geo_Cluster"),
        ("zip_region", "MLP_ZIP"),
        ("risk_band", "MLP_Risk_Band"),
        ("city_embedding", "MLP_CityEmbedding"),
    ],
}

round3_logistic_params = {
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": 42,
}

round3_xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "scale_pos_weight": (y_train_val == 0).sum() / (y_train_val == 1).sum(),
    "random_state": 42,
    "n_jobs": -1,
}

round3_mlp_params = {
    "hidden_dim": 64,
    "batch_size": 64,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "max_epochs": 80,
    "patience": 8,
    "val_size": 0.15,
    "threshold": 0.5,
    "random_state": 42,
    "verbose": False,
}

round3_embedding_params = {
    "city_column": "City",
    "geo_drop_columns": ("Zip Code", "Latitude", "Longitude", "Lat Long"),
    "embedding_dim": None,
}

In [23]:
round3_results_by_model = {}
round3_fold_results_by_model = {}

for model_name, strategy_specs in round3_strategy_specs.items():
    print(f"=== ROUND 3: {model_name} ===")

    model_results, model_fold_results = evaluate_round3_model_strategies(
        model_name,
        strategy_specs,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        preprocessor=preprocessor,
        fe_params=round3_fe_params,
        y_reference=y_train_val,
        metrics=round3_metrics,
        target_smoothing=20.0,
        logistic_params=round3_logistic_params,
        xgb_params=round3_xgb_params,
        mlp_params=round3_mlp_params,
        embedding_params=round3_embedding_params,
    )

    round3_results_by_model[model_name] = model_results
    round3_fold_results_by_model[model_name] = model_fold_results

=== ROUND 3: LogisticRegression ===
=== ROUND 3: XGBoost ===
=== ROUND 3: MLP ===


Resultados por modelo: LogisticRegression

In [24]:
display(round3_results_by_model["LogisticRegression"].round(4))

,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s,base_model,strategy
0,LogisticRegression_ZIP,0.9394,0.9759,0.9312,0.7835,0.8508,0.0620,0.0224,LogisticRegression,zip_region
1,LogisticRegression_Geo_Cluster,0.9390,0.9757,0.9266,0.7842,0.8493,0.3338,0.0235,LogisticRegression,geo_cluster
2,LogisticRegression_Frequency,0.9390,0.9758,0.9266,0.7807,0.8472,0.0445,0.0208,LogisticRegression,frequency
3,LogisticRegression_Target,0.9221,0.9689,0.8777,0.7792,0.8253,0.0781,0.0205,LogisticRegression,target
4,LogisticRegression_Risk_Band,0.9147,0.9644,0.8716,0.7712,0.8181,0.0432,0.0213,LogisticRegression,risk_band


Resultados por modelo: XGBoost

In [25]:
display(round3_results_by_model["XGBoost"].round(4))

,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s,base_model,strategy
0,XGBoost_Geo_Cluster,0.9527,0.9809,0.8838,0.8468,0.8648,0.1028,0.0367,XGBoost,geo_cluster
1,XGBoost_ZIP,0.9515,0.9808,0.8823,0.8493,0.8654,0.0950,0.0304,XGBoost,zip_region
2,XGBoost_Frequency,0.9493,0.9799,0.8776,0.8478,0.8623,0.0689,0.0274,XGBoost,frequency
3,XGBoost_Risk_Band,0.9356,0.9731,0.8494,0.8374,0.8432,0.0755,0.0292,XGBoost,risk_band
4,XGBoost_Target,0.9330,0.9719,0.8234,0.8559,0.8392,0.0705,0.0277,XGBoost,target


Resultados por modelo: MLP

In [26]:
display(round3_results_by_model["MLP"].round(4))

,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s,base_model,strategy
0,MLP_ZIP,0.9371,0.9750,0.9281,0.7633,0.8369,1.2879,0.0300,MLP,zip_region
1,MLP_Geo_Cluster,0.9370,0.9748,0.9312,0.7634,0.8379,0.9980,0.0291,MLP,geo_cluster
2,MLP_Frequency,0.9362,0.9747,0.9266,0.7643,0.8372,1.0547,0.0237,MLP,frequency
3,MLP_Target,0.9202,0.9683,0.8929,0.7713,0.8273,1.0446,0.0239,MLP,target
4,MLP_CityEmbedding,0.9163,0.9667,0.9075,0.7453,0.8172,1.1698,0.0277,MLP,city_embedding
5,MLP_Risk_Band,0.9044,0.9598,0.8746,0.7543,0.8093,1.1654,0.0286,MLP,risk_band


Resultado consolidado do Round 3

In [22]:
round3_results_all = (
    pd.concat(round3_results_by_model.values(), ignore_index=True)
    .sort_values(["base_model", "pr_auc_mean"], ascending=[True, False])
    .reset_index(drop=True)
)

display(round3_results_all.round(4))

,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s,base_model,strategy
0,LogisticRegression_ZIP,0.9394,0.9759,0.9312,0.7835,0.8508,0.0595,0.0213,LogisticRegression,zip_region
1,LogisticRegression_Geo_Cluster,0.9390,0.9757,0.9266,0.7842,0.8493,0.3275,0.0235,LogisticRegression,geo_cluster
2,LogisticRegression_Frequency,0.9390,0.9758,0.9266,0.7807,0.8472,0.0474,0.0225,LogisticRegression,frequency
3,LogisticRegression_Target,0.9221,0.9689,0.8777,0.7792,0.8253,0.0749,0.0196,LogisticRegression,target
4,LogisticRegression_Risk_Band,0.9147,0.9644,0.8716,0.7712,0.8181,0.0416,0.0204,LogisticRegression,risk_band
5,MLP_Geo_Cluster,0.9373,0.9750,0.9304,0.7697,0.8416,1.0658,0.0299,MLP,geo_cluster
6,MLP_ZIP,0.9361,0.9746,0.9228,0.7723,0.8399,1.1560,0.0259,MLP,zip_region
7,MLP_Frequency,0.9358,0.9746,0.9327,0.7645,0.8396,1.0348,0.0239,MLP,frequency
8,MLP_Target,0.9194,0.9679,0.8914,0.7694,0.8247,1.0896,0.0230,MLP,target
9,MLP_CityEmbedding,0.9171,0.9663,0.9182,0.7326,0.8135,1.0504,0.0232,MLP,city_embedding


### Conclusão

As variáveis geográficas não demonstraram ganho robusto o suficiente para justificar sua incorporação no pipeline final. Embora algumas estratégias, como zip_region na Regressão Logística e geo_cluster no XGBoost, tenham produzido pequenas melhoras marginais, os ganhos foram muito discretos e inconsistentes entre os modelos. Na MLP, inclusive, abordagens mais sofisticadas como CityEmbedding elevaram o recall, mas com perda relevante de precisão. Considerando o aumento de complexidade, o risco de instabilidade e o baixo retorno incremental observado, a decisão é não incluir as variáveis geográficas na versão final do conjunto de features.

### Round 4 - Feature Selection Exploratória

Objetivo desta rodada:
- explorar faixas promissoras de `K` para `LogisticRegression`, `XGBoost` e `MLP`;
- comparar `f_classif` e `mutual_info_classif` sem `GridSearchCV`;
- consolidar o ranking por `PR-AUC > ROC-AUC > Recall`.

#### SelectKBest

In [28]:
assert "City" in X_train_val.columns, "City precisa estar presente em X_train_val para esta rodada."
assert "Churn Score" in X_train_val.columns, "Churn Score precisa estar presente em X_train_val para esta rodada."
assert "CLTV" not in X_train_val.columns, "CLTV deve permanecer fora de X_train_val como metadata."

round4_logger = get_logger("round4_feature_selection")
round4_sort_columns = ["pr_auc_mean", "roc_auc_mean", "recall_mean"]
round4_sort_ascending = [False, False, False]


def rank_round4_results(df):
    return df.sort_values(round4_sort_columns, ascending=round4_sort_ascending).reset_index(drop=True)


selector_label_map = {
    f_classif: "f_classif",
    mutual_info_classif: "mutual_info_classif",
}
selector_specs = [
    (f_classif, "f_classif"),
    (mutual_info_classif, "mutual_info_classif"),
]

round4_fe_params = dict(
    drop_churn_score=False,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)

round4_fe = FeatureEngineerTransformer(**round4_fe_params)
round4_geo = GeoTransformer(strategy="drop")

X_train_val_round4 = round4_fe.fit_transform(X_train_val, y_train_val)
X_train_val_round4 = round4_geo.fit_transform(X_train_val_round4, y_train_val)

processed_feature_names = get_processed_feature_names(
    preprocessor,
    X_train_val_round4,
    y_train_val,
)

k_grid = build_k_grid(
    n_features_processed=len(processed_feature_names),
    min_k=10,
    include_all=False,
    step=10,
)

round4_logger.info(
    "Round 4 | total de features processadas=%d | k_grid=%s",
    len(processed_feature_names),
    k_grid,
)


def build_round4_estimators():
    return {
        "LogisticRegression": Pipeline(
            [
                ("fe", FeatureEngineerTransformer(**round4_fe_params)),
                ("geo", GeoTransformer(strategy="drop")),
                ("prep", preprocessor),
                ("selector", SelectKBest()),
                ("scaler", StandardScaler(with_mean=False)),
                (
                    "model",
                    LogisticRegression(
                        max_iter=1000,
                        class_weight="balanced",
                        random_state=42,
                    ),
                ),
            ]
        ),
        "XGBoost": Pipeline(
            [
                ("fe", FeatureEngineerTransformer(**round4_fe_params)),
                ("geo", GeoTransformer(strategy="drop")),
                ("prep", preprocessor),
                ("selector", SelectKBest()),
                (
                    "model",
                    XGBClassifier(
                        objective="binary:logistic",
                        eval_metric="logloss",
                        scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
                        random_state=42,
                        n_jobs=-1,
                    ),
                ),
            ]
        ),
        "MLP": Pipeline(
            [
                ("fe", FeatureEngineerTransformer(**round4_fe_params)),
                ("geo", GeoTransformer(strategy="drop")),
                ("prep", preprocessor),
                ("selector", SelectKBest()),
                ("scaler", StandardScaler(with_mean=False)),
                (
                    "model",
                    MLPClassifierWrapper(
                        hidden_dim=64,
                        batch_size=64,
                        lr=1e-3,
                        weight_decay=1e-5,
                        dropout=0.0,
                        max_epochs=80,
                        patience=16,
                        min_delta=1e-3,
                        val_size=0.15,
                        threshold=0.5,
                        random_state=42,
                        verbose=False,
                    ),
                ),
            ]
        ),
    }


round4_rows = []
feature_selection_searches = {}
selected_feature_logs = {}
round4_fold_results = {}

for model_name, estimator in build_round4_estimators().items():
    round4_logger.info("Round 4 | iniciando avaliacao do modelo=%s", model_name)

    for score_func, selector_name in selector_specs:
        for k_value in k_grid:
            round4_logger.info(
                "Round 4 | model=%s | selector=%s | k=%s",
                model_name,
                selector_name,
                k_value,
            )

            estimator_iter = clone(estimator)
            estimator_iter.set_params(
                selector__score_func=score_func,
                selector__k=k_value,
            )

            cv_res = cross_validate(
                estimator=estimator_iter,
                X=X_train_val,
                y=y_train_val,
                cv=cv,
                scoring=scoring,
                n_jobs=1,
                return_train_score=False,
            )

            experiment_name = f"{model_name}__{selector_name}__k_{k_value}"
            fold_df = pd.DataFrame(
                {
                    "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
                    "pr_auc": cv_res["test_pr_auc"],
                    "roc_auc": cv_res["test_roc_auc"],
                    "recall": cv_res["test_recall"],
                    "precision": cv_res["test_precision"],
                    "f1": cv_res["test_f1"],
                    "fit_time_s": cv_res["fit_time"],
                    "score_time_s": cv_res["score_time"],
                }
            )
            round4_fold_results[experiment_name] = fold_df

            fitted_full = clone(estimator_iter).fit(X_train_val, y_train_val)
            feature_selection_searches[experiment_name] = fitted_full
            selected_features = extract_selected_feature_names(
                fitted_full,
                processed_feature_names,
                selector_step="selector",
            )
            selected_feature_logs[experiment_name] = selected_features

            round4_rows.append(
                {
                    "model": model_name,
                    "selector": selector_name,
                    "k": int(k_value),
                    "pr_auc_mean": cv_res["test_pr_auc"].mean(),
                    "pr_auc_std": cv_res["test_pr_auc"].std(),
                    "roc_auc_mean": cv_res["test_roc_auc"].mean(),
                    "recall_mean": cv_res["test_recall"].mean(),
                    "precision_mean": cv_res["test_precision"].mean(),
                    "f1_mean": cv_res["test_f1"].mean(),
                    "fit_time_mean_s": cv_res["fit_time"].mean(),
                    "score_time_mean_s": cv_res["score_time"].mean(),
                }
            )

results_fs = rank_round4_results(pd.DataFrame(round4_rows))

round4_results_by_model = {
    model_name: rank_round4_results(results_fs[results_fs["model"] == model_name].copy())
    for model_name in results_fs["model"].unique()
}

results_fs_logreg = round4_results_by_model["LogisticRegression"].copy()
results_fs_xgb = round4_results_by_model["XGBoost"].copy()
results_fs_mlp = round4_results_by_model["MLP"].copy()

fs_analysis_cols = [
    "selector",
    "k",
    "pr_auc_mean",
    "pr_auc_std",
    "roc_auc_mean",
    "recall_mean",
    "precision_mean",
    "f1_mean",
    "fit_time_mean_s",
    "score_time_mean_s",
]

fs_logreg_analysis = results_fs_logreg[fs_analysis_cols].copy()
fs_xgb_analysis = results_fs_xgb[fs_analysis_cols].copy()
fs_mlp_analysis = results_fs_mlp[fs_analysis_cols].copy()

round4_k_candidates = {}
for model_name, df_model in round4_results_by_model.items():
    top_df = df_model.head(min(5, len(df_model))).copy()
    round4_k_candidates[model_name] = sorted(top_df["k"].astype(int).unique().tolist())
    round4_logger.info(
        "Round 4 | model=%s | faixa promissora de k=%s",
        model_name,
        round4_k_candidates[model_name],
    )

assert len(round4_results_by_model) == 3, "A rodada de feature selection deve retornar exatamente 3 modelos."

print("=== FEATURE SELECTION: LOGISTIC REGRESSION ===")
display(results_fs_logreg.round(4))

print("=== FEATURE SELECTION: XGBOOST ===")
display(results_fs_xgb.round(4))

print("=== FEATURE SELECTION: MLP ===")
display(results_fs_mlp.round(4))


2026-04-30 19:25:58 [INFO] round4_feature_selection: Round 4 | total de features processadas=58 | k_grid=[10, 20, 30, 40, 50, 58]
2026-04-30 19:25:58 [INFO] round4_feature_selection: Round 4 | iniciando avaliacao do modelo=LogisticRegression


2026-04-30 19:25:58 [INFO] round4_feature_selection: Round 4 | model=LogisticRegression | selector=f_classif | k=10
2026-04-30 19:25:58 [INFO] round4_feature_selection: Round 4 | model=LogisticRegression | selector=f_classif | k=20
2026-04-30 19:25:59 [INFO] round4_feature_selection: Round 4 | model=LogisticRegression | selector=f_classif | k=30
2026-04-30 19:25:59 [INFO] round4_feature_selection: Round 4 | model=LogisticRegression | selector=f_classif | k=40
2026-04-30 19:26:00 [INFO] round4_feature_selection: Round 4 | model=LogisticRegression | selector=f_classif | k=50
2026-04-30 19:26:00 [INFO] round4_feature_selection: Round 4 | model=LogisticRegression | selector=f_classif | k=58
2026-04-30 19:26:01 [INFO] round4_feature_selection: Round 4 | model=LogisticRegression | selector=mutual_info_classif | k=10
2026-04-30 19:26:03 [INFO] round4_feature_selection: Round 4 | model=LogisticRegression | selector=mutual_info_classif | k=20
2026-04-30 19:26:06 [INFO] round4_feature_selection:

,model,selector,k,pr_auc_mean,pr_auc_std,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression,f_classif,30,0.9398,0.0108,0.9763,0.9342,0.7805,0.8502,0.0548,0.0203
1,LogisticRegression,f_classif,40,0.9398,0.0111,0.9762,0.9274,0.7838,0.8493,0.0637,0.0206
2,LogisticRegression,mutual_info_classif,40,0.9397,0.0111,0.9761,0.9289,0.7851,0.8507,0.4503,0.0208
3,LogisticRegression,mutual_info_classif,30,0.9397,0.0106,0.9763,0.9335,0.7825,0.8510,0.4545,0.0200
4,LogisticRegression,f_classif,50,0.9397,0.0108,0.9760,0.9289,0.7832,0.8496,0.0637,0.0208
5,LogisticRegression,f_classif,20,0.9395,0.0104,0.9762,0.9312,0.7805,0.8490,0.0428,0.0192
6,LogisticRegression,f_classif,58,0.9393,0.0105,0.9759,0.9297,0.7827,0.8497,0.0476,0.0205
7,LogisticRegression,mutual_info_classif,58,0.9393,0.0105,0.9759,0.9297,0.7827,0.8497,0.4385,0.0218
8,LogisticRegression,mutual_info_classif,50,0.9392,0.0106,0.9759,0.9297,0.7833,0.8500,0.4442,0.0296
9,LogisticRegression,mutual_info_classif,20,0.9380,0.0093,0.9757,0.9312,0.7771,0.8469,0.4501,0.0210


=== FEATURE SELECTION: XGBOOST ===


,model,selector,k,pr_auc_mean,pr_auc_std,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,XGBoost,f_classif,30,0.9529,0.0089,0.9812,0.8799,0.8434,0.8611,0.0704,0.0286
1,XGBoost,f_classif,20,0.9526,0.0080,0.9808,0.8953,0.8457,0.8695,0.0653,0.0279
2,XGBoost,mutual_info_classif,50,0.9524,0.0096,0.9809,0.8777,0.8442,0.8605,0.4935,0.0295
3,XGBoost,f_classif,50,0.9519,0.0086,0.9806,0.8784,0.8449,0.8613,0.0797,0.0290
4,XGBoost,f_classif,58,0.9512,0.0112,0.9802,0.8738,0.8404,0.8566,0.0833,0.0295
5,XGBoost,mutual_info_classif,58,0.9512,0.0112,0.9802,0.8738,0.8404,0.8566,0.4886,0.0294
6,XGBoost,mutual_info_classif,30,0.9511,0.0078,0.9806,0.8815,0.8471,0.8638,0.4809,0.0303
7,XGBoost,f_classif,40,0.9509,0.0085,0.9803,0.8792,0.8463,0.8623,0.0757,0.0286
8,XGBoost,mutual_info_classif,40,0.9505,0.0090,0.9801,0.8761,0.8448,0.8600,0.4869,0.0287
9,XGBoost,mutual_info_classif,20,0.9487,0.0073,0.9795,0.8845,0.8380,0.8605,0.4753,0.0279


=== FEATURE SELECTION: MLP ===


,model,selector,k,pr_auc_mean,pr_auc_std,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,MLP,f_classif,20,0.9387,0.0102,0.9762,0.9335,0.7737,0.8455,1.7583,0.0250
1,MLP,mutual_info_classif,40,0.9386,0.0116,0.9756,0.9304,0.7733,0.8440,2.2627,0.0285
2,MLP,f_classif,50,0.9383,0.0108,0.9753,0.9289,0.7715,0.8420,1.6263,0.0257
3,MLP,f_classif,30,0.9376,0.0122,0.9755,0.9319,0.7710,0.8427,2.0353,0.0235
4,MLP,mutual_info_classif,30,0.9371,0.0111,0.9752,0.9335,0.7708,0.8432,2.4027,0.0284
5,MLP,f_classif,40,0.9371,0.0099,0.9750,0.9296,0.7695,0.8413,1.5982,0.0248
6,MLP,mutual_info_classif,50,0.9367,0.0111,0.9749,0.9220,0.7687,0.8376,2.1299,0.0290
7,MLP,f_classif,58,0.9360,0.0088,0.9746,0.9312,0.7631,0.8383,1.5422,0.0252
8,MLP,mutual_info_classif,58,0.9359,0.0093,0.9746,0.9296,0.7694,0.8411,2.3440,0.0295
9,MLP,mutual_info_classif,20,0.9357,0.0108,0.9747,0.9327,0.7707,0.8428,2.4072,0.0259


In [30]:
selector_metric_cols = [
    "pr_auc_mean",
    "pr_auc_std",
    "roc_auc_mean",
    "recall_mean",
    "precision_mean",
    "f1_mean",
    "fit_time_mean_s",
    "score_time_mean_s",
]


def summarize_selector_performance(df, model_name):
    selector_summary = (
        df.groupby("selector", as_index=False)[selector_metric_cols]
        .mean()
        .sort_values(
            ["pr_auc_mean", "roc_auc_mean", "recall_mean"],
            ascending=[False, False, False],
        )
        .reset_index(drop=True)
    )

    print(f"=== DESEMPENHO MÉDIO POR SELECTOR: {model_name} ===")
    display(selector_summary.round(4))
    return selector_summary


selector_summary_logreg = summarize_selector_performance(
    results_fs_logreg,
    "LogisticRegression",
)
selector_summary_xgb = summarize_selector_performance(
    results_fs_xgb,
    "XGBoost",
)
selector_summary_mlp = summarize_selector_performance(
    results_fs_mlp,
    "MLP",
)



=== DESEMPENHO MÉDIO POR SELECTOR: LogisticRegression ===


,selector,pr_auc_mean,pr_auc_std,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,f_classif,0.9383,0.0103,0.9755,0.9280,0.7789,0.8467,0.0513,0.0201
1,mutual_info_classif,0.9380,0.0101,0.9754,0.9288,0.7791,0.8471,0.4444,0.0222


=== DESEMPENHO MÉDIO POR SELECTOR: XGBoost ===


,selector,pr_auc_mean,pr_auc_std,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,f_classif,0.9511,0.0086,0.9803,0.8812,0.8410,0.8604,0.0728,0.0285
1,mutual_info_classif,0.9501,0.0085,0.9800,0.8796,0.8407,0.8595,0.4857,0.0290


=== DESEMPENHO MÉDIO POR SELECTOR: MLP ===


,selector,pr_auc_mean,pr_auc_std,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,f_classif,0.9361,0.0103,0.9747,0.9308,0.7683,0.8410,1.8164,0.0248
1,mutual_info_classif,0.9356,0.0109,0.9745,0.9295,0.7685,0.8405,2.3437,0.0276


In [33]:
def summarize_k_performance(df, model_name):
    k_summary = (
        df.groupby("k")
        .agg(
            n_configs=("k", "size"),
            pr_auc_mean_avg=("pr_auc_mean", "mean"),
            pr_auc_mean_median=("pr_auc_mean", "median"),
            pr_auc_mean_max=("pr_auc_mean", "max"),
            pr_auc_mean_min=("pr_auc_mean", "min"),
            pr_auc_std_avg=("pr_auc_std", "mean"),
            roc_auc_mean_avg=("roc_auc_mean", "mean"),
            roc_auc_mean_median=("roc_auc_mean", "median"),
            recall_mean_avg=("recall_mean", "mean"),
            recall_mean_median=("recall_mean", "median"),
            precision_mean_avg=("precision_mean", "mean"),
            f1_mean_avg=("f1_mean", "mean"),
            fit_time_mean_avg_s=("fit_time_mean_s", "mean"),
            score_time_mean_avg_s=("score_time_mean_s", "mean"),
        )
        .sort_values(
            ["pr_auc_mean_avg", "roc_auc_mean_avg", "recall_mean_avg"],
            ascending=[False, False, False],
        )
        .reset_index()
    )

    print(f"=== RESUMO DETALHADO POR K: {model_name} ===")
    display(k_summary.round(4))
    return k_summary


k_summary_logreg = summarize_k_performance(
    results_fs_logreg,
    "LogisticRegression",
)

k_summary_xgb = summarize_k_performance(
    results_fs_xgb,
    "XGBoost",
)

k_summary_mlp = summarize_k_performance(
    results_fs_mlp,
    "MLP",
)


=== RESUMO DETALHADO POR K: LogisticRegression ===


,k,n_configs,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_mean_max,pr_auc_mean_min,pr_auc_std_avg,roc_auc_mean_avg,roc_auc_mean_median,recall_mean_avg,recall_mean_median,precision_mean_avg,f1_mean_avg,fit_time_mean_avg_s,score_time_mean_avg_s
0,30,2,0.9398,0.9398,0.9398,0.9397,0.0107,0.9763,0.9763,0.9339,0.9339,0.7815,0.8506,0.2546,0.0202
1,40,2,0.9397,0.9397,0.9398,0.9397,0.0111,0.9762,0.9762,0.9281,0.9281,0.7844,0.8500,0.2570,0.0207
2,50,2,0.9394,0.9394,0.9397,0.9392,0.0107,0.9759,0.9759,0.9293,0.9293,0.7832,0.8498,0.2539,0.0252
3,58,2,0.9393,0.9393,0.9393,0.9393,0.0105,0.9759,0.9759,0.9297,0.9297,0.7827,0.8497,0.2431,0.0212
4,20,2,0.9388,0.9388,0.9395,0.9380,0.0098,0.9759,0.9759,0.9312,0.9312,0.7788,0.8480,0.2465,0.0201
5,10,2,0.9321,0.9321,0.9322,0.9319,0.0085,0.9725,0.9725,0.9182,0.9182,0.7634,0.8335,0.2321,0.0195


=== RESUMO DETALHADO POR K: XGBoost ===


,k,n_configs,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_mean_max,pr_auc_mean_min,pr_auc_std_avg,roc_auc_mean_avg,roc_auc_mean_median,recall_mean_avg,recall_mean_median,precision_mean_avg,f1_mean_avg,fit_time_mean_avg_s,score_time_mean_avg_s
0,50,2,0.9521,0.9521,0.9524,0.9519,0.0091,0.9807,0.9807,0.8780,0.8780,0.8446,0.8609,0.2866,0.0293
1,30,2,0.9520,0.9520,0.9529,0.9511,0.0084,0.9809,0.9809,0.8807,0.8807,0.8453,0.8625,0.2757,0.0295
2,58,2,0.9512,0.9512,0.9512,0.9512,0.0112,0.9802,0.9802,0.8738,0.8738,0.8404,0.8566,0.2860,0.0295
3,40,2,0.9507,0.9507,0.9509,0.9505,0.0087,0.9802,0.9802,0.8777,0.8777,0.8455,0.8611,0.2813,0.0286
4,20,2,0.9506,0.9506,0.9526,0.9487,0.0076,0.9802,0.9802,0.8899,0.8899,0.8419,0.8650,0.2703,0.0279
5,10,2,0.9469,0.9469,0.9472,0.9466,0.0060,0.9785,0.9785,0.8823,0.8823,0.8274,0.8537,0.2755,0.0279


=== RESUMO DETALHADO POR K: MLP ===


,k,n_configs,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_mean_max,pr_auc_mean_min,pr_auc_std_avg,roc_auc_mean_avg,roc_auc_mean_median,recall_mean_avg,recall_mean_median,precision_mean_avg,f1_mean_avg,fit_time_mean_avg_s,score_time_mean_avg_s
0,40,2,0.9378,0.9378,0.9386,0.9371,0.0108,0.9753,0.9753,0.9300,0.9300,0.7714,0.8427,1.9304,0.0267
1,50,2,0.9375,0.9375,0.9383,0.9367,0.0110,0.9751,0.9751,0.9254,0.9254,0.7701,0.8398,1.8781,0.0274
2,30,2,0.9374,0.9374,0.9376,0.9371,0.0116,0.9754,0.9754,0.9327,0.9327,0.7709,0.8430,2.2190,0.0260
3,20,2,0.9372,0.9372,0.9387,0.9357,0.0105,0.9754,0.9754,0.9331,0.9331,0.7722,0.8441,2.0827,0.0254
4,58,2,0.9360,0.9360,0.9360,0.9359,0.0090,0.9746,0.9746,0.9304,0.9304,0.7662,0.8397,1.9431,0.0274
5,10,2,0.9294,0.9294,0.9298,0.9291,0.0107,0.9719,0.9719,0.9293,0.9293,0.7596,0.8351,2.4269,0.0244


In [34]:
def summarize_k_best_case(df, model_name):
    k_best = (
        df.sort_values(
            ["pr_auc_mean", "roc_auc_mean", "recall_mean"],
            ascending=[False, False, False],
        )
        .groupby("k", as_index=False)
        .first()
        .sort_values(
            ["pr_auc_mean", "roc_auc_mean", "recall_mean"],
            ascending=[False, False, False],
        )
        .reset_index(drop=True)
    )

    print(f"=== MELHOR CONFIGURAÇÃO POR K: {model_name} ===")
    display(k_best.round(4))
    return k_best


k_best_logreg = summarize_k_best_case(results_fs_logreg, "LogisticRegression")
k_best_xgb = summarize_k_best_case(results_fs_xgb, "XGBoost")
k_best_mlp = summarize_k_best_case(results_fs_mlp, "MLP")


=== MELHOR CONFIGURAÇÃO POR K: LogisticRegression ===


,k,model,selector,pr_auc_mean,pr_auc_std,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,30,LogisticRegression,f_classif,0.9398,0.0108,0.9763,0.9342,0.7805,0.8502,0.0548,0.0203
1,40,LogisticRegression,f_classif,0.9398,0.0111,0.9762,0.9274,0.7838,0.8493,0.0637,0.0206
2,50,LogisticRegression,f_classif,0.9397,0.0108,0.9760,0.9289,0.7832,0.8496,0.0637,0.0208
3,20,LogisticRegression,f_classif,0.9395,0.0104,0.9762,0.9312,0.7805,0.8490,0.0428,0.0192
4,58,LogisticRegression,f_classif,0.9393,0.0105,0.9759,0.9297,0.7827,0.8497,0.0476,0.0205
5,10,LogisticRegression,mutual_info_classif,0.9322,0.0085,0.9727,0.9197,0.7639,0.8344,0.4289,0.0200


=== MELHOR CONFIGURAÇÃO POR K: XGBoost ===


,k,model,selector,pr_auc_mean,pr_auc_std,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,30,XGBoost,f_classif,0.9529,0.0089,0.9812,0.8799,0.8434,0.8611,0.0704,0.0286
1,20,XGBoost,f_classif,0.9526,0.0080,0.9808,0.8953,0.8457,0.8695,0.0653,0.0279
2,50,XGBoost,mutual_info_classif,0.9524,0.0096,0.9809,0.8777,0.8442,0.8605,0.4935,0.0295
3,58,XGBoost,f_classif,0.9512,0.0112,0.9802,0.8738,0.8404,0.8566,0.0833,0.0295
4,40,XGBoost,f_classif,0.9509,0.0085,0.9803,0.8792,0.8463,0.8623,0.0757,0.0286
5,10,XGBoost,f_classif,0.9472,0.0061,0.9785,0.8807,0.8251,0.8518,0.0620,0.0276


=== MELHOR CONFIGURAÇÃO POR K: MLP ===


,k,model,selector,pr_auc_mean,pr_auc_std,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,20,MLP,f_classif,0.9387,0.0102,0.9762,0.9335,0.7737,0.8455,1.7583,0.0250
1,40,MLP,mutual_info_classif,0.9386,0.0116,0.9756,0.9304,0.7733,0.8440,2.2627,0.0285
2,50,MLP,f_classif,0.9383,0.0108,0.9753,0.9289,0.7715,0.8420,1.6263,0.0257
3,30,MLP,f_classif,0.9376,0.0122,0.9755,0.9319,0.7710,0.8427,2.0353,0.0235
4,58,MLP,f_classif,0.9360,0.0088,0.9746,0.9312,0.7631,0.8383,1.5422,0.0252
5,10,MLP,mutual_info_classif,0.9298,0.0114,0.9720,0.9289,0.7585,0.8344,2.5155,0.0241


#### Conclusão

Com base nos resultados do Round 4, a leitura mais coerente é a seguinte.

No `XGBoost`, o melhor bloco de desempenho está entre `20` e `50` features, com maior concentração de bons resultados em `20–30`. O melhor resultado individual apareceu em `k=30` com `f_classif`, mas `k=20` ficou praticamente empatado em `PR-AUC` e ainda apresentou métricas competitivas nas demais dimensões. Por isso, a faixa candidata mais consistente para o modelo é `20–30`, podendo ser ampliada até `20–50` caso queiramos manter uma janela mais flexível na etapa seguinte.

Na `LogisticRegression`, os resultados mostram um platô bastante estável entre `20` e `58` features, com diferenças muito pequenas entre as melhores configurações. Ainda assim, a região de `30–40` concentra os melhores desempenhos médios e representa uma escolha mais enxuta sem perda relevante de qualidade. Assim, a faixa candidata mais coerente para esse modelo é `20–50`, com preferência prática por `30–40`.

Na `MLP`, `k=10` piora de forma mais evidente e `k=58` já começa a perder força, enquanto o melhor bloco de desempenho fica entre `20` e `50`, especialmente em `20–40`. O melhor resultado individual foi obtido com `k=20` usando `f_classif`, enquanto `k=40` com `mutual_info_classif` apareceu logo em seguida, mostrando que essa segunda alternativa é competitiva, mas não dominante. Com isso, a faixa candidata mais adequada para a `MLP` é `20–40`, podendo ser ampliada até `50` se quisermos preservar mais flexibilidade.

Também há um sinal importante na comparação entre seletores. De forma geral, `f_classif` se mostrou tão bom quanto ou melhor que `mutual_info_classif` na maior parte dos cenários, além de apresentar menor custo computacional, especialmente em `XGBoost` e `LogisticRegression`. Na `MLP`, `mutual_info_classif` aparece como alternativa competitiva em alguns pontos específicos, mas sem evidência suficiente para justificar sua adoção como padrão. Assim, a decisão metodologicamente mais coerente é manter `f_classif` como seletor preferencial nesta rodada, utilizando `mutual_info_classif` apenas como referência complementar em casos específicos.

Em síntese, as faixas candidatas de `K` para a próxima etapa ficam definidas como:
- `XGBoost`: `20–30`, com possibilidade de ampliar para `20–50`
- `LogisticRegression`: `20–50`, com preferência por `30–40`
- `MLP`: `20–40`, com possibilidade de ampliar para `20–50`

Dessa forma, o Round 4 cumpre seu papel exploratório ao reduzir o espaço de busca de forma orientada pelos resultados, sem fixar prematuramente um único valor de `K`.


#### L1-Based Selection - Regressão Logística

**Regras**

- Limitar a regularização para não passar de 10 features (mínimo exigido pelo projeto) → Early Stopping

In [24]:
import logging
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from src.features.feature_engineer_transformer import FeatureEngineerTransformer
from src.features.geo_transformer import GeoTransformer


# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------
logger = logging.getLogger("round4_l1_logreg")
logger.setLevel(logging.INFO)
logger.propagate = False

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%H:%M:%S",
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def build_l1_logreg_pipeline(C, fe_params):
    return Pipeline([
        ("fe", FeatureEngineerTransformer(**fe_params)),
        ("geo", GeoTransformer(strategy="drop")),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            solver="saga",
            l1_ratio=1.0,
            C=C,
            max_iter=5000,
            class_weight="balanced",
            random_state=42,
        )),
    ])


def get_nonzero_feature_names(fitted_pipeline):
    feature_names = fitted_pipeline.named_steps["prep"].get_feature_names_out()
    coefs = fitted_pipeline.named_steps["model"].coef_.ravel()
    nonzero_mask = coefs != 0
    selected_features = np.asarray(feature_names)[nonzero_mask].tolist()
    return selected_features, coefs


# ------------------------------------------------------------
# Grid de C
# Mais a  esquerda = regularização mais fraca
# Mais a  direita = regularização mais forte
# A busca para quando < 10 features restarem
# ------------------------------------------------------------
c_grid = [
    10.0, 7.5, 5.0, 3.0, 2.0, 1.0,
    0.75, 0.5, 0.3, 0.2, 0.1,
    0.075, 0.05, 0.04, 0.03, 0.02, 0.01,
    0.0075, 0.005, 0.004, 0.003, 0.002, 0.001,
]

min_features_threshold = 10
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]

rows = []
l1_search_logs = {}
l1_search_fold_results = {}

for C in c_grid:
    logger.info("Iniciando avaliação com L1 LogisticRegression | C=%.5f", C)

    estimator = build_l1_logreg_pipeline(
        C=C,
        fe_params=round4_fe_params,
    )

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
        return_estimator=True,
    )

    fold_df = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    l1_search_fold_results[C] = fold_df

    # Refit no conjunto completo para extrair as features finais selecionadas
    fitted_full = clone(estimator).fit(X_train_val, y_train_val)
    selected_features, coefs = get_nonzero_feature_names(fitted_full)
    n_selected = len(selected_features)

    row = {
        "model": "LogisticRegression_L1",
        "C": C,
        "n_selected_features": n_selected,
        "pr_auc_mean": cv_res["test_pr_auc"].mean(),
        "roc_auc_mean": cv_res["test_roc_auc"].mean(),
        "recall_mean": cv_res["test_recall"].mean(),
        "precision_mean": cv_res["test_precision"].mean(),
        "f1_mean": cv_res["test_f1"].mean(),
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    }
    rows.append(row)

    l1_search_logs[C] = {
        "selected_features": selected_features,
        "coefficients": coefs,
        "metrics": row,
    }

    logger.info(
        (
            "Resultado | C=%.5f | selected=%d | "
            "PR-AUC=%.4f | ROC-AUC=%.4f | Recall=%.4f | Precision=%.4f | F1=%.4f"
        ),
        C,
        n_selected,
        row["pr_auc_mean"],
        row["roc_auc_mean"],
        row["recall_mean"],
        row["precision_mean"],
        row["f1_mean"],
    )

    logger.info("Features selecionadas (%d): %s", n_selected, selected_features)

    if n_selected < min_features_threshold:
        logger.info(
            (
                "Early stopping acionado: número de features selecionadas "
                "(%d) ficou abaixo do limite de %d."
            ),
            n_selected,
            min_features_threshold,
        )
        break

results_l1_fs = (
    pd.DataFrame(rows)
    .sort_values(["pr_auc_mean", "roc_auc_mean", "recall_mean"], ascending=[False, False, False])
    .reset_index(drop=True)
)

display(results_l1_fs.round(4))


17:25:31 | INFO | Iniciando avaliação com L1 LogisticRegression | C=10.00000
17:25:39 | INFO | Resultado | C=10.00000 | selected=47 | PR-AUC=0.9388 | ROC-AUC=0.9758 | Recall=0.9289 | Precision=0.7821 | F1=0.8490
17:25:39 | INFO | Features selecionadas (47): ['cat__Gender_Female', 'cat__Senior Citizen_No', 'cat__Partner_Yes', 'cat__Dependents_Yes', 'cat__Phone Service_No', 'cat__Phone Service_Yes', 'cat__Multiple Lines_No', 'cat__Multiple Lines_No phone service', 'cat__Internet Service_DSL', 'cat__Internet Service_Fiber optic', 'cat__Internet Service_No', 'cat__Online Security_No internet service', 'cat__Online Security_Yes', 'cat__Online Backup_No internet service', 'cat__Online Backup_Yes', 'cat__Device Protection_No', 'cat__Device Protection_No internet service', 'cat__Device Protection_Yes', 'cat__Tech Support_No', 'cat__Tech Support_No internet service', 'cat__Tech Support_Yes', 'cat__Streaming TV_No', 'cat__Streaming TV_No internet service', 'cat__Streaming TV_Yes', 'cat__Streamin

,model,C,n_selected_features,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression_L1,0.0750,28,0.9400,0.9762,0.9327,0.7782,0.8483,0.5021,0.0205
1,LogisticRegression_L1,0.1000,28,0.9400,0.9762,0.9289,0.7786,0.8469,0.6151,0.0208
2,LogisticRegression_L1,0.0400,26,0.9399,0.9762,0.9350,0.7717,0.8454,0.3739,0.0212
3,LogisticRegression_L1,0.2000,29,0.9399,0.9761,0.9281,0.7819,0.8486,1.1900,0.0242
4,LogisticRegression_L1,0.0500,26,0.9399,0.9762,0.9335,0.7730,0.8455,0.3938,0.0208
5,LogisticRegression_L1,0.3000,34,0.9397,0.9760,0.9274,0.7827,0.8488,1.4714,0.0216
6,LogisticRegression_L1,0.0300,24,0.9396,0.9762,0.9358,0.7690,0.8440,0.3342,0.0208
7,LogisticRegression_L1,0.5000,35,0.9396,0.9759,0.9258,0.7815,0.8474,1.6140,0.0219
8,LogisticRegression_L1,0.7500,34,0.9394,0.9759,0.9266,0.7832,0.8487,2.3136,0.0251
9,LogisticRegression_L1,1.0000,36,0.9393,0.9758,0.9274,0.7828,0.8488,2.5883,0.0207


In [25]:
thr = results_l1_fs['pr_auc_mean'].quantile(0.80)
top_trials_l1 = results_l1_fs[results_l1_fs['pr_auc_mean'] >= thr]
top_trials_l1 = top_trials_l1.sort_values('pr_auc_mean', ascending=False).reset_index(drop=True)
top_trials_l1.describe()

,C,n_selected_features,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
count,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.093000,27.400000,0.939954,0.976187,0.931644,0.776674,0.846946,0.614963,0.021508
std,0.064187,1.341641,0.000044,0.000050,0.002992,0.004214,0.001495,0.335585,0.001546
min,0.040000,26.000000,0.939915,0.976107,0.928124,0.771739,0.845402,0.373854,0.020468
25%,0.050000,26.000000,0.939920,0.976181,0.928891,0.772975,0.845527,0.393758,0.020826
50%,0.075000,28.000000,0.939936,0.976196,0.932716,0.778221,0.846907,0.502130,0.020835
75%,0.100000,28.000000,0.939983,0.976208,0.933480,0.778557,0.848318,0.615066,0.021174
max,0.200000,29.000000,0.940017,0.976241,0.935009,0.781876,0.848576,1.190005,0.024237


### Conclusão

Na `LogisticRegression`, os resultados com `SelectKBest` já indicavam um comportamento bastante estável entre diferentes seletores e quantidades de features. Para refinar essa leitura, foi realizada uma etapa adicional de seleção por regularização L1, mais alinhada à própria estrutura do modelo.

Os resultados confirmaram a existência de um platô de desempenho muito consistente, com os melhores valores concentrados no intervalo entre `C=0.04` e `C=0.20`, selecionando aproximadamente `26` a `29` features. Dentro dessa região, as diferenças de `PR-AUC`, `ROC-AUC` e `F1` foram mínimas, indicando que a regressão logística é pouco sensível a pequenas variações no conjunto de features, desde que permaneça nessa faixa.

Dessa forma, a leitura metodologicamente mais coerente é abandonar uma faixa ampla de `K` para a `LogisticRegression` e adotar como referência uma região mais estável e parcimoniosa, em torno de `26–29` features. Entre as configurações avaliadas, `C=0.075` se destaca por combinar o melhor `PR-AUC` com um conjunto enxuto de `28` features, sendo uma escolha natural como ponto de partida.


## Fine Tuning de Hiperparametros em 2 Etapas

**Objetivo:**
- fazer uma exploracao inicial com `RandomizedSearchCV` otimizando `PR-AUC`;
- gerar um dataset de resultados para a `MLP` e outro para o `XGBoost`;
- usar os melhores trials para definir uma faixa reduzida de busca na segunda etapa com `Optuna`.

### Etapa 1 - RandomizedSearchCV: MLP

In [ ]:
mlp_logger = get_logger("round4_mlp_random_search")

mlp_param_distributions = {
    "selector__score_func": [f_classif],
    "selector__k": [20, 30, 40],
    "model__activation": ["relu", "gelu", "tanh"],
    "model__hidden_dim": [32, 64, 128],
    "model__dropout": [0.0, 0.1, 0.2, 0.3],
    "model__lr": [1e-4, 3e-4, 1e-3, 3e-3],
    "model__weight_decay": [0.0, 1e-6, 1e-5, 1e-4],
    "model__batch_size": [32, 64, 128],
}

mlp_base_pipeline = Pipeline(
    [
        ("fe", FeatureEngineerTransformer(**round4_fe_params)),
        ("geo", GeoTransformer(strategy="drop")),
        ("prep", preprocessor),
        ("selector", SelectKBest()),
        ("scaler", StandardScaler(with_mean=False)),
        (
            "model",
            MLPClassifierWrapper(
                output_dim=1,
                max_epochs=80,
                patience=16,
                min_delta=1e-3,
                threshold=0.5,
                val_size=0.15,
                random_state=42,
                verbose=False,
            ),
        ),
    ]
)

mlp_random_search = RandomizedSearchCV(
    estimator=mlp_base_pipeline,
    param_distributions=mlp_param_distributions,
    n_iter=100,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=1,
    random_state=42,
    return_train_score=False,
    verbose=1,
)
mlp_random_search.fit(X_train_val, y_train_val)

mlp_rows = []
mlp_cv_results = mlp_random_search.cv_results_
for trial_idx, params in enumerate(mlp_cv_results["params"], start=1):
    selector_func = params["selector__score_func"]
    mlp_rows.append(
        {
            "trial_id": trial_idx,
            "k": int(params["selector__k"]),
            "activation": params["model__activation"],
            "hidden_dim": int(params["model__hidden_dim"]),
            "dropout": float(params["model__dropout"]),
            "lr": float(params["model__lr"]),
            "weight_decay": float(params["model__weight_decay"]),
            "batch_size": int(params["model__batch_size"]),
            "pr_auc_mean": mlp_cv_results["mean_test_pr_auc"][trial_idx - 1],
            "pr_auc_std": mlp_cv_results["std_test_pr_auc"][trial_idx - 1],
            "roc_auc_mean": mlp_cv_results["mean_test_roc_auc"][trial_idx - 1],
            "recall_mean": mlp_cv_results["mean_test_recall"][trial_idx - 1],
            "precision_mean": mlp_cv_results["mean_test_precision"][trial_idx - 1],
            "f1_mean": mlp_cv_results["mean_test_f1"][trial_idx - 1],
            "fit_time_mean_s": mlp_cv_results["mean_fit_time"][trial_idx - 1],
            "score_time_mean_s": mlp_cv_results["mean_score_time"][trial_idx - 1],
        }
    )

results_mlp_random_search = (
    pd.DataFrame(mlp_rows)
    .sort_values(["pr_auc_mean", "roc_auc_mean", "recall_mean"], ascending=[False, False, False])
    .reset_index(drop=True)
)
results_mlp_grid = results_mlp_random_search.copy()

best_mlp_estimator = mlp_random_search.best_estimator_
best_mlp_params = mlp_random_search.best_params_
best_mlp_selected_features = extract_selected_feature_names(
    best_mlp_estimator,
    processed_feature_names,
    selector_step="selector",
)
best_mlp_result = results_mlp_random_search.iloc[0].to_dict()

mlp_logger.info("RandomizedSearchCV da MLP finalizado | top PR-AUC=%.4f", best_mlp_result["pr_auc_mean"])
mlp_logger.info("Melhores parâmetros da MLP: %s", best_mlp_params)
mlp_logger.info("Features selecionadas na melhor configuracao da MLP: %s", best_mlp_selected_features)

display(results_mlp_random_search.head(20).round(4))


Fitting 5 folds for each of 100 candidates, totalling 500 fits
2026-04-30 20:25:12 [INFO] round4_mlp_random_search: RandomizedSearchCV da MLP finalizado | top PR-AUC=0.9394
2026-04-30 20:25:12 [INFO] round4_mlp_random_search: Melhores parâmetros da MLP: {'selector__score_func': <function f_classif at 0x00000187A72B5850>, 'selector__k': 40, 'model__weight_decay': 1e-06, 'model__lr': 0.001, 'model__hidden_dim': 32, 'model__dropout': 0.3, 'model__batch_size': 128, 'model__activation': 'tanh'}
2026-04-30 20:25:12 [INFO] round4_mlp_random_search: Features selecionadas na melhor configuracao da MLP: ['cat__Senior Citizen_No', 'cat__Partner_No', 'cat__Partner_Yes', 'cat__Dependents_No', 'cat__Dependents_Yes', 'cat__Internet Service_Fiber optic', 'cat__Internet Service_No', 'cat__Online Security_No', 'cat__Online Security_No internet service', 'cat__Online Security_Yes', 'cat__Online Backup_No', 'cat__Online Backup_No internet service', 'cat__Device Protection_No', 'cat__Device Protection_No i

,trial_id,selector,k,activation,hidden_dim,dropout,lr,weight_decay,batch_size,pr_auc_mean,pr_auc_std,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,5,f_classif,40,tanh,32,0.3,0.0010,0.0000,128,0.9394,0.0117,0.9761,0.9327,0.7741,0.8455,1.6425,0.0253
1,40,f_classif,20,tanh,32,0.2,0.0010,0.0000,64,0.9393,0.0110,0.9762,0.9342,0.7743,0.8461,2.1604,0.0241
2,82,f_classif,40,tanh,64,0.1,0.0030,0.0000,128,0.9390,0.0119,0.9758,0.9273,0.7773,0.8451,0.8864,0.0234
3,38,f_classif,40,tanh,32,0.1,0.0010,0.0000,128,0.9389,0.0118,0.9759,0.9312,0.7725,0.8440,1.7656,0.0239
4,53,f_classif,40,tanh,64,0.3,0.0010,0.0001,64,0.9389,0.0118,0.9760,0.9388,0.7668,0.8434,2.1251,0.0321
5,60,f_classif,30,tanh,32,0.3,0.0010,0.0000,32,0.9389,0.0130,0.9759,0.9281,0.7751,0.8440,3.2878,0.0278
6,17,f_classif,40,tanh,64,0.2,0.0003,0.0000,32,0.9388,0.0116,0.9759,0.9335,0.7721,0.8445,4.0145,0.0250
7,1,f_classif,40,tanh,64,0.0,0.0003,0.0001,128,0.9388,0.0122,0.9759,0.9281,0.7810,0.8478,1.9574,0.0249
8,10,f_classif,40,tanh,64,0.1,0.0003,0.0000,128,0.9387,0.0122,0.9759,0.9327,0.7740,0.8455,2.1397,0.0227
9,18,f_classif,40,tanh,128,0.3,0.0001,0.0000,32,0.9387,0.0117,0.9758,0.9296,0.7776,0.8463,5.7411,0.0285


In [36]:
results_mlp_random_search.shape

(100, 22)

In [37]:
assert best_mlp_result is not None, "Nenhum resultado foi encontrado na busca da MLP."
assert best_mlp_params is not None, "best_mlp_params não foi definido."
assert best_mlp_estimator is not None, "best_mlp_estimator não foi definido."

best_mlp_summary = pd.DataFrame([best_mlp_result]).round(4)
display(best_mlp_summary)

print("A definição do espaço inicial do Optuna para a MLP será consolidada após a análise exploratória do RandomizedSearchCV.")

,trial_id,selector,k,activation,hidden_dim,dropout,lr,weight_decay,batch_size,pr_auc_mean,pr_auc_std,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,5,f_classif,40,tanh,32,0.3,0.001,0.0,128,0.9394,0.0117,0.9761,0.9327,0.7741,0.8455,1.6425,0.0253


A definição do espaço inicial do Optuna para a MLP será consolidada após a análise exploratória do RandomizedSearchCV.


### Análise exploratória dos resultados do RandomizedSearchCV

Fluxo adotado:

- rankear por `PR-AUC`;
- selecionar o `Top 20%`;
- filtrar os trials mais estáveis com base em `PR-AUC_std`;
- resumir os parâmetros com `value_counts`, `describe` e `groupby`.

Aqui vamos priorizar:

1. `lr`
2. `weight_decay`
3. `dropout`
4. `hidden_dim`
5. `batch_size`
6. `k`
7. `activation`

#### Separando os top20 modelos mais promissores

In [49]:
mlp_rs = results_mlp_random_search.sort_values(
    ["pr_auc_mean", "roc_auc_mean", "recall_mean"],
    ascending=[False, False, False],
).reset_index(drop=True)

top_thr = mlp_rs["pr_auc_mean"].quantile(0.80)
top20 = mlp_rs[mlp_rs["pr_auc_mean"] >= top_thr].copy()

print("Total:", len(mlp_rs))
print("Top 20%:", len(top20))



Total: 100
Top 20%: 20


#### Analisando a frequência dos parâmetros no top20 modelos treinados

In [ ]:
def summarize_discrete_lift(df_all, df_top, col):
    total = df_all[col].value_counts(normalize=True).rename("share_total")
    top = df_top[col].value_counts(normalize=True).rename("share_top")
    out = (
        pd.concat([total, top], axis=1)
        .fillna(0)
        .reset_index()
        .rename(columns={"index": col})
    )
    out["lift"] = out["share_top"] / out["share_total"]
    return out.sort_values(["lift", "share_top"], ascending=[False, False])

for col in ["k", "activation", "hidden_dim", "batch_size", "lr", "weight_decay", "dropout"]:
    print(f"=== {col} ===")
    display(summarize_discrete_lift(mlp_rs, top20, col).round(4))


=== selector ===


,selector,share_total,share_top,lift
0,f_classif,1.0,1.0,1.0


=== k ===


,k,share_total,share_top,lift
0,40,0.43,0.60,1.3953
2,20,0.21,0.25,1.1905
1,30,0.36,0.15,0.4167


=== activation ===


,activation,share_total,share_top,lift
1,tanh,0.34,0.80,2.3529
2,gelu,0.31,0.15,0.4839
0,relu,0.35,0.05,0.1429


=== hidden_dim ===


,hidden_dim,share_total,share_top,lift
0,64,0.34,0.45,1.3235
2,32,0.32,0.35,1.0938
1,128,0.34,0.20,0.5882


=== batch_size ===


,batch_size,share_total,share_top,lift
2,32,0.22,0.25,1.1364
1,64,0.36,0.35,0.9722
0,128,0.42,0.40,0.9524


#### Conclusão

Na análise de frequência dos hiperparâmetros discretos dentro do subconjunto de melhores trials estáveis, o valor de `lift` ajuda a identificar quais configurações aparecem no topo com maior frequência do que seria esperado pela sua presença no conjunto total. Ele responde: **“esse valor aparece no topo mais do que seria esperado pelo acaso?”**

Interpretação:

- lift = 1
    esse valor aparece no top exatamente na mesma proporção que aparece no conjunto total
    sinal neutro

- lift > 1
    esse valor aparece mais no top do que no total
    sinal positivo, possivelmente promissor

- lift < 1
    esse valor aparece menos no top do que no total
    sinal negativo

Para `k`, o valor `40` foi o que apresentou o sinal mais forte, com `share_top = 0.60` e `lift = 1.3953`. Isso indica que ele está sobre-representado entre os melhores resultados estáveis e, portanto, surge como o valor mais promissor nessa etapa. O `k=20` também aparece com sinal positivo (`lift = 1.1905`), embora de forma mais moderada. Já `k=30` ficou sub-representado no topo (`lift = 0.4167`), o que sugere menor força relativa dentro desse conjunto filtrado.

Na ativação, o destaque é bastante claro para `tanh`, que aparece em `80%` dos melhores trials estáveis, apesar de representar apenas `34%` do total, resultando em `lift = 2.3529`. Esse é um sinal forte de associação positiva com os melhores resultados observados. Em contraste, `gelu` ficou sub-representado (`lift = 0.4839`) e `relu` praticamente não aparece no topo (`lift = 0.1429`), indicando que, nesta rodada, ambas perdem prioridade frente ao `tanh`.

Para `hidden_dim`, o valor `64` foi o mais favorecido, com `lift = 1.3235`, sugerindo que essa largura aparece com mais frequência entre os melhores trials do que no conjunto total. O valor `32` aparece quase de forma neutra (`lift = 1.0938`), o que indica que ainda pode ser mantido como alternativa plausível. Já `128` ficou sub-representado (`lift = 0.5882`), perdendo força como candidato principal para a etapa seguinte.

No caso de `batch_size`, os três valores ficaram relativamente próximos, o que mostra um sinal menos discriminante. Ainda assim, `32` teve a melhor relação (`lift = 1.1364`), indicando leve sobre-representação entre os melhores trials estáveis. O `64` ficou praticamente neutro (`lift = 0.9722`) e o `128` também apresentou comportamento quase neutro (`lift = 0.9524`). Isso sugere que `batch_size` não é, nesta análise, um fator tão decisivo quanto `k`, `activation` ou `hidden_dim`.

De forma geral, essa tabela aponta como sinais mais fortes para a próxima etapa: `k=40`, ativação `tanh` e `hidden_dim=64`. O `k=20` e o `hidden_dim=32` permanecem como alternativas razoáveis, enquanto `k=30`, `relu` e `hidden_dim=128` perdem prioridade dentro do espaço de busca.


#### Análisando o ganho marginal

In [ ]:
def summarize_param_effect(df, col):
    return (
        df.groupby(col)
        .agg(
            n_trials=(col, "size"),
            pr_auc_mean_avg=("pr_auc_mean", "mean"),
            pr_auc_mean_median=("pr_auc_mean", "median"),
            pr_auc_std_avg=("pr_auc_std", "mean"),
            roc_auc_mean_avg=("roc_auc_mean", "mean"),
            recall_mean_avg=("recall_mean", "mean"),
            fit_time_mean_avg_s=("fit_time_mean_s", "mean"),
        )
        .sort_values(
            ["pr_auc_mean_avg", "roc_auc_mean_avg", "recall_mean_avg"],
            ascending=[False, False, False],
        )
        .reset_index()
    )

for col in ["k", "activation", "hidden_dim", "batch_size", "dropout", "lr", "weight_decay"]:
    print(f"=== {col} ===")
    display(summarize_param_effect(mlp_rs, col).round(4))


=== selector ===


,selector,n_trials,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_std_avg,roc_auc_mean_avg,recall_mean_avg,fit_time_mean_avg_s
0,f_classif,100,0.9336,0.9364,0.0131,0.9737,0.9309,2.5042


=== k ===


,k,n_trials,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_std_avg,roc_auc_mean_avg,recall_mean_avg,fit_time_mean_avg_s
0,40,43,0.9357,0.9370,0.0123,0.9746,0.9310,2.5506
1,30,36,0.9342,0.9364,0.0137,0.9740,0.9321,2.4950
2,20,21,0.9285,0.9354,0.0135,0.9713,0.9285,2.4250


=== activation ===


...

,dropout,n_trials,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_std_avg,roc_auc_mean_avg,recall_mean_avg,fit_time_mean_avg_s
0,0.0,20,0.9349,0.9367,0.0127,0.9742,0.9313,2.1773
1,0.3,26,0.9337,0.9358,0.0133,0.9738,0.9314,2.7434
2,0.1,29,0.9332,0.9366,0.0130,0.9735,0.9305,2.5140
3,0.2,25,0.9330,0.9366,0.0132,0.9734,0.9305,2.5056


=== lr ===


,lr,n_trials,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_std_avg,roc_auc_mean_avg,recall_mean_avg,fit_time_mean_avg_s
0,0.0010,19,0.9374,0.9372,0.0120,0.9754,0.9321,2.3549
1,0.0003,35,0.9364,0.9370,0.0128,0.9750,0.9325,2.8404
2,0.0030,27,0.9362,0.9362,0.0121,0.9749,0.9322,1.4927
3,0.0001,19,0.9211,0.9257,0.0160,0.9677,0.9248,3.4717


=== weight_decay ===


,weight_decay,n_trials,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_std_avg,roc_auc_mean_avg,recall_mean_avg,fit_time_mean_avg_s
0,0.0000,28,0.9343,0.9366,0.0132,0.9740,0.9310,2.8076
1,0.0000,19,0.9341,0.9364,0.0130,0.9739,0.9319,2.8451
2,0.0000,21,0.9333,0.9371,0.0129,0.9735,0.9304,2.3120
3,0.0001,32,0.9330,0.9363,0.0130,0.9734,0.9304,2.1625


#### Conclusão

Na análise de ganho marginal, o objetivo é avaliar como cada hiperparâmetro se comporta quando observado isoladamente, isto é, olhando para a média, mediana, estabilidade e custo de treino agregados por valor.

Para `k`, o melhor desempenho médio aparece em `k=40`, com `pr_auc_mean_avg = 0.9356`, além da melhor mediana e da menor variabilidade relativa entre os três valores testados. O `k=30` também se mantém competitivo, com desempenho próximo e recall ligeiramente superior, enquanto `k=20` fica claramente abaixo em `PR-AUC` e `ROC-AUC`. Isso reforça a leitura de que a região mais promissora está concentrada entre `30` e `40`, com vantagem prática para `40`.

Na ativação, a leitura é mais sutil. Pela média, `relu` aparece na frente, seguido de `gelu` e `tanh`. No entanto, as diferenças são pequenas, e `tanh` apresenta a maior mediana de `PR-AUC`, o que sugere que ela concentra alguns dos melhores resultados individuais, mesmo não liderando a média global. Em termos metodológicos, isso indica que não há dominância absoluta de uma única ativação, mas sim um equilíbrio entre desempenho médio e potencial de pico. Ainda assim, `relu` e `tanh` surgem como os candidatos mais relevantes, enquanto `gelu` permanece como alternativa intermediária.

Para `hidden_dim`, o valor `64` se destaca com a melhor média e mediana de `PR-AUC`, além de boa estabilidade. O `128` fica logo atrás, com desempenho próximo e custo de treino ligeiramente menor, enquanto `32` aparece claramente inferior em média, mediana e estabilidade. Isso sugere que a região mais adequada para a largura da rede está entre `64` e `128`, com prioridade para `64`.

Em `batch_size`, o valor `64` entrega o melhor equilíbrio geral entre desempenho e custo. O `32` é competitivo em `PR-AUC` e apresenta o melhor `recall`, mas exige mais tempo de treino. Já `128` é o mais barato computacionalmente, porém com perda perceptível de qualidade média. Portanto, `64` aparece como escolha principal, enquanto `32` pode ser mantido como alternativa quando a prioridade for maximizar desempenho e não tempo de execução.

No caso do `dropout`, as diferenças entre os valores são relativamente pequenas. O `0.0` apresenta a melhor média e a melhor mediana de `PR-AUC`, além de menor custo computacional, enquanto `0.1`, `0.2` e `0.3` permanecem próximos, sem ruptura evidente de desempenho. Isso sugere que o efeito marginal do `dropout` é fraco nesta rodada: ele continua relevante como regularização, mas sem um valor claramente dominante. Assim, ainda faz sentido mantê-lo no espaço de busca, porém em uma faixa curta e controlada.

Para `lr`, o sinal é muito mais forte. O valor `0.001` lidera em média e mediana de `PR-AUC`, com boa estabilidade e custo moderado. O `0.0003` e o `0.0030` continuam competitivos, mas `0.0001` se mostra claramente inferior em todas as métricas principais e ainda com maior tempo de treino. Isso indica que a região mais promissora da taxa de aprendizado está entre `3e-4` e `1e-3`, com `1e-3` como centro principal da busca.

Por fim, em `weight_decay`, as diferenças são pequenas, mas existe uma leve vantagem para os valores mais baixos. `1e-5` apresenta a melhor média, `1e-6` vem logo em seguida, e `0.0` tem a melhor mediana com menor custo que os dois anteriores. O valor `1e-4` fica levemente atrás. Essa leitura sugere que o `weight_decay` deve continuar próximo de zero, sem necessidade de explorar regularizações mais altas nesta etapa.

De forma geral, a análise de ganho marginal aponta para o seguinte desenho mais promissor para a próxima etapa: `k` na região de `30–40`, com preferência por `40`; `hidden_dim` em `64`, mantendo `128` como alternativa; `batch_size` em `64`, com `32` como opção complementar; `dropout` em faixa curta, sem necessidade de fixação; `lr` centrado em `1e-3`, com extensão até `3e-4`; e `weight_decay` restrito a valores muito baixos, próximos de zero.


#### testando algumas interações entre as variáveis

- k x activation -> análise da entrada do modelo
- hidden_dim x dropout -> análise de capacidade/complexidade vs regularização
- lr x weight_decay -> otimização vs regularização
- batch_size x lr -> dinâmica de treino

In [ ]:
def summarize_pair(df, col_a, col_b):
    return (
        df.groupby([col_a, col_b])
        .agg(
            n_trials=("pr_auc_mean", "size"),
            pr_auc_mean_avg=("pr_auc_mean", "mean"),
            pr_auc_mean_median=("pr_auc_mean", "median"),
            pr_auc_std_avg=("pr_auc_std", "mean"),
        )
        .reset_index()
        .sort_values(
            ["pr_auc_mean_avg", "pr_auc_mean_median"],
            ascending=[False, False],
        )
    )

print("=== Interação: hidden_dim x dropout ===")
display(summarize_pair(top20, "hidden_dim", "dropout").round(4))
print("=== Interação: lr x weight_decay ===")
display(summarize_pair(top20, "lr", "weight_decay").round(4))
print("=== Interação: k x activation ===")
display(summarize_pair(top20, "k", "activation").round(4))
print("=== Interação: batch_size x lr ===")
display(summarize_pair(top20, "batch_size", "lr").round(4))


=== Interação: hidden_dim x dropout ===


,hidden_dim,dropout,n_trials,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_std_avg
2,32,0.3,1,0.9394,0.9394,0.0117
1,32,0.2,1,0.9393,0.9393,0.0110
6,64,0.3,1,0.9389,0.9389,0.0118
5,64,0.2,1,0.9388,0.9388,0.0116
8,128,0.3,1,0.9387,0.9387,0.0117
4,64,0.1,1,0.9385,0.9385,0.0107
7,128,0.0,1,0.9385,0.9385,0.0109
3,64,0.0,2,0.9384,0.9384,0.0114
0,32,0.0,1,0.9382,0.9382,0.0113


=== Interação: lr x weight_decay ===


,lr,weight_decay,n_trials,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_std_avg
4,0.0010,0.0000,2,0.9393,0.9393,0.0114
5,0.0010,0.0001,1,0.9389,0.9389,0.0118
2,0.0003,0.0000,2,0.9387,0.9387,0.0115
0,0.0001,0.0000,1,0.9387,0.9387,0.0117
1,0.0003,0.0000,2,0.9385,0.9385,0.0108
3,0.0003,0.0000,1,0.9383,0.9383,0.0114
6,0.0030,0.0001,1,0.9382,0.9382,0.0113


=== Interação: k x activation ===


,k,activation,n_trials,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_std_avg
1,20,tanh,2,0.9389,0.9389,0.0109
3,40,tanh,5,0.9389,0.9388,0.0115
2,40,gelu,2,0.9384,0.9384,0.0114
0,20,gelu,1,0.9382,0.9382,0.0113


=== Interação: batch_size x lr ===


,batch_size,lr,n_trials,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_std_avg
6,128,0.0010,1,0.9394,0.9394,0.0117
4,64,0.0010,2,0.9391,0.9391,0.0114
1,32,0.0003,1,0.9388,0.9388,0.0116
0,32,0.0001,1,0.9387,0.9387,0.0117
5,128,0.0003,1,0.9386,0.9386,0.0114
3,64,0.0003,3,0.9384,0.9385,0.0110
2,32,0.0030,1,0.9382,0.9382,0.0113


#### Conclusão

Na análise de interações, o objetivo é verificar se alguns hiperparâmetros passam a funcionar melhor quando combinados entre si, em vez de serem interpretados isoladamente. Como o subconjunto utilizado aqui é pequeno, os resultados devem ser lidos como sinais direcionais, e não como evidência definitiva.

Na interação entre `hidden_dim` e `dropout`, os melhores resultados aparecem com `hidden_dim=32` combinado a `dropout=0.3` e `dropout=0.2`, seguidos por combinações de `hidden_dim=64` com `dropout=0.3` e `0.2`. Isso sugere que níveis moderados ou altos de `dropout` continuam compatíveis com bons desempenhos tanto para redes menores quanto médias. Ao mesmo tempo, `hidden_dim=64` com `dropout=0.0` aparece duas vezes e permanece competitivo, indicando que a largura intermediária da rede é mais robusta e consegue funcionar bem mesmo com menor regularização explícita. Já `hidden_dim=128` só aparece em combinações pontuais e não mostra força suficiente para ganhar prioridade nesta etapa.

Na interação entre `lr` e `weight_decay`, a melhor combinação observada foi `lr=0.001` com `weight_decay=0.0`, seguida de `lr=0.001` com `weight_decay=0.0001`. Em seguida aparecem combinações com `lr=0.0003` e `weight_decay` muito baixo. Isso reforça que a melhor região de otimização está centrada em `lr=1e-3`, com regularização L2 muito pequena ou nula. Em contrapartida, combinações com `weight_decay` mais elevado não aparecem entre as melhores, o que confirma a leitura anterior de que esse parâmetro deve permanecer próximo de zero.

Na interação entre `k` e `activation`, o melhor desempenho aparece tanto em `k=20` com `tanh` quanto em `k=40` com `tanh`, sendo esta última a combinação com maior recorrência (`n_trials=5`). As combinações com `gelu` também aparecem, mas em segundo plano, e sempre abaixo das equivalentes com `tanh`. Isso indica que `tanh` se adapta bem às duas regiões mais promissoras de entrada (`k=20` e `k=40`), enquanto `gelu` pode ser mantida apenas como alternativa complementar. A ativação `relu` não aparece entre as melhores interações, perdendo força nesta leitura combinada.

Por fim, na interação entre `batch_size` e `lr`, a melhor combinação individual foi `batch_size=128` com `lr=0.001`, seguida de `batch_size=64` com `lr=0.001`. Ainda assim, `batch_size=64` com `lr=0.001` parece mais confiável metodologicamente, por aparecer com maior recorrência e permanecer muito próximo do topo. As combinações com `lr=0.0003` continuam competitivas, mas ligeiramente abaixo. Já `lr=0.003` aparece apenas de forma pontual, sem evidência suficiente para se tornar região central da busca.

De forma geral, a análise de interações reforça alguns sinais que já haviam aparecido nas análises marginais: a região mais promissora está concentrada em `lr=0.001`, `weight_decay` próximo de zero, ativação `tanh`, `k` em `20` ou `40`, e `hidden_dim` preferencialmente em `64`, com `32` como alternativa válida. Também sugere que `batch_size=64` continua sendo a escolha mais equilibrada, enquanto `128` pode ser mantido como opção complementar quando combinado com `lr=0.001`.


### Grid Paramétrico Reduzido para o Optuna

In [ ]:
mlp_optuna_search_space = {
    "selector__k": {"low": 35, "high": 45, "step": 1},
    "model__activation": ["tanh"],
    "model__hidden_dim": {"low": 32, "high": 64, "step": 2},
    "model__dropout": {"low": 0.0, "high": 0.3, "step": 0.1},
    "model__lr": {"low": 3e-4, "high": 1e-3, "log": True},
    "model__weight_decay": [0.0, 1e-6, 1e-5],
    "model__batch_size": [64, 80, 96, 112, 128],
}

### Etapa 1 - RandomizedSearchCV: XGBoost

In [55]:
xgb_logger = get_logger("round4_xgb_random_search")

xgb_param_distributions = {
    "model__n_estimators": [200, 300, 500, 700],
    "model__max_depth": [3, 5, 7, 9, 12, 20],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__min_child_weight": [1, 3, 5, 7],
    "model__subsample": [0.7, 0.85, 1.0],
    "model__colsample_bytree": [0.7, 0.85, 1.0],
    "model__gamma": [0.0, 0.1, 0.3, 0.5],
    "model__reg_alpha": [0.0, 0.1, 0.5, 1.0],
    "model__reg_lambda": [1.0, 2.0, 5.0, 10.0],
}

xgb_base_pipeline = Pipeline(
    [
        ("fe", FeatureEngineerTransformer(**round4_fe_params)),
        ("geo", GeoTransformer(strategy="drop")),
        ("prep", preprocessor),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

xgb_random_search = RandomizedSearchCV(
    estimator=xgb_base_pipeline,
    param_distributions=xgb_param_distributions,
    n_iter=100,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=1,
    random_state=42,
    return_train_score=False,
    verbose=1,
)

xgb_random_search.fit(X_train_val, y_train_val)

xgb_rows = []
xgb_cv_results = xgb_random_search.cv_results_

for trial_idx, params in enumerate(xgb_cv_results["params"], start=1):
    xgb_rows.append(
        {
            "trial_id": trial_idx,
            "n_estimators": int(params["model__n_estimators"]),
            "max_depth": int(params["model__max_depth"]),
            "learning_rate": float(params["model__learning_rate"]),
            "min_child_weight": int(params["model__min_child_weight"]),
            "subsample": float(params["model__subsample"]),
            "colsample_bytree": float(params["model__colsample_bytree"]),
            "gamma": float(params["model__gamma"]),
            "reg_alpha": float(params["model__reg_alpha"]),
            "reg_lambda": float(params["model__reg_lambda"]),
            "pr_auc_mean": xgb_cv_results["mean_test_pr_auc"][trial_idx - 1],
            "pr_auc_std": xgb_cv_results["std_test_pr_auc"][trial_idx - 1],
            "roc_auc_mean": xgb_cv_results["mean_test_roc_auc"][trial_idx - 1],
            "recall_mean": xgb_cv_results["mean_test_recall"][trial_idx - 1],
            "precision_mean": xgb_cv_results["mean_test_precision"][trial_idx - 1],
            "f1_mean": xgb_cv_results["mean_test_f1"][trial_idx - 1],
            "fit_time_mean_s": xgb_cv_results["mean_fit_time"][trial_idx - 1],
            "score_time_mean_s": xgb_cv_results["mean_score_time"][trial_idx - 1],
        }
    )

results_xgb_random_search = (
    pd.DataFrame(xgb_rows)
    .sort_values(
        ["pr_auc_mean", "roc_auc_mean", "recall_mean"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

results_xgb_grid = results_xgb_random_search.copy()

best_xgb_estimator = xgb_random_search.best_estimator_
best_xgb_params = xgb_random_search.best_params_
best_xgb_result = results_xgb_random_search.iloc[0].to_dict()

xgb_logger.info(
    "RandomizedSearchCV do XGBoost finalizado | top PR-AUC=%.4f",
    best_xgb_result["pr_auc_mean"],
)
xgb_logger.info("Melhores parâmetros do XGBoost: %s", best_xgb_params)

display(results_xgb_random_search.head(20).round(4))


Fitting 5 folds for each of 100 candidates, totalling 500 fits
2026-04-30 21:53:24 [INFO] round4_xgb_random_search: RandomizedSearchCV do XGBoost finalizado | top PR-AUC=0.9604
2026-04-30 21:53:24 [INFO] round4_xgb_random_search: Melhores parâmetros do XGBoost: {'model__subsample': 1.0, 'model__reg_lambda': 10.0, 'model__reg_alpha': 0.1, 'model__n_estimators': 300, 'model__min_child_weight': 1, 'model__max_depth': 3, 'model__learning_rate': 0.05, 'model__gamma': 0.3, 'model__colsample_bytree': 0.85}


,trial_id,n_estimators,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,pr_auc_mean,pr_auc_std,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,5,300,3,0.05,1,1.00,0.85,0.3,0.1,10.0,0.9604,0.0092,0.9844,0.9472,0.8064,0.8709,0.0941,0.0271
1,24,200,3,0.05,5,1.00,0.70,0.5,0.5,2.0,0.9601,0.0088,0.9843,0.9480,0.8027,0.8692,0.0890,0.0296
2,55,500,3,0.03,1,1.00,0.70,0.3,1.0,1.0,0.9598,0.0091,0.9842,0.9472,0.8079,0.8718,0.1559,0.0295
3,51,300,5,0.03,3,1.00,0.85,0.3,0.0,5.0,0.9598,0.0094,0.9841,0.9419,0.8090,0.8703,0.1301,0.0296
4,84,300,3,0.05,5,1.00,0.70,0.0,1.0,5.0,0.9597,0.0086,0.9841,0.9465,0.8034,0.8689,0.0991,0.0281
5,38,700,5,0.01,7,1.00,0.85,0.5,0.0,10.0,0.9596,0.0090,0.9841,0.9480,0.8043,0.8701,0.2839,0.0334
6,20,500,3,0.05,3,1.00,0.70,0.5,0.1,5.0,0.9595,0.0082,0.9841,0.9465,0.8115,0.8736,0.1193,0.0277
7,99,200,3,0.03,3,0.70,0.70,0.1,0.1,2.0,0.9595,0.0085,0.9841,0.9495,0.8040,0.8705,0.0774,0.0276
8,34,300,3,0.05,1,0.85,1.00,0.0,0.0,10.0,0.9594,0.0086,0.9841,0.9427,0.8142,0.8735,0.1441,0.0305
9,86,700,3,0.01,7,0.70,1.00,0.5,0.0,10.0,0.9593,0.0088,0.9840,0.9480,0.8054,0.8707,0.1934,0.0288


In [56]:
results_xgb_random_search.shape

(100, 18)

### Análise exploratória dos resultados do RandomizedSearchCV

#### Separando os top20 modelos mais promissores

In [59]:
xgb_rs = results_xgb_random_search.sort_values(
    ["pr_auc_mean", "roc_auc_mean", "recall_mean"],
    ascending=[False, False, False],
).reset_index(drop=True)

top_thr = xgb_rs["pr_auc_mean"].quantile(0.80)
top20 = xgb_rs[xgb_rs["pr_auc_mean"] >= top_thr].copy()

print("Total:", len(xgb_rs))
print("Top 20%:", len(top20))



Total: 100
Top 20%: 20


#### Analisando a frequência dos parâmetros no top20 modelos treinados

In [61]:
for col in ["n_estimators", "max_depth", "learning_rate", "min_child_weight", "subsample", "colsample_bytree", "gamma", "reg_alpha", "reg_lambda"]:
    print(f"=== {col} ===")
    display(summarize_discrete_lift(xgb_rs, top20, col).round(4))


=== n_estimators ===


,n_estimators,share_total,share_top,lift
1,700,0.25,0.30,1.2000
3,500,0.22,0.25,1.1364
0,300,0.29,0.30,1.0345
2,200,0.24,0.15,0.6250


=== max_depth ===


,max_depth,share_total,share_top,lift
0,3,0.21,0.7,3.3333
3,5,0.15,0.2,1.3333
4,7,0.15,0.1,0.6667
1,20,0.21,0.0,0.0000
2,9,0.17,0.0,0.0000
5,12,0.11,0.0,0.0000


=== learning_rate ===


...

,gamma,share_total,share_top,lift
0,0.5,0.30,0.5,1.6667
2,0.0,0.25,0.2,0.8000
1,0.3,0.28,0.2,0.7143
3,0.1,0.17,0.1,0.5882


=== reg_alpha ===


,reg_alpha,share_total,share_top,lift
0,0.0,0.30,0.40,1.3333
2,1.0,0.22,0.25,1.1364
3,0.5,0.19,0.20,1.0526
1,0.1,0.29,0.15,0.5172


=== reg_lambda ===


,reg_lambda,share_total,share_top,lift
1,10.0,0.29,0.40,1.3793
3,2.0,0.18,0.20,1.1111
2,5.0,0.23,0.25,1.0870
0,1.0,0.30,0.15,0.5000


#### Análisando o ganho marginal

In [63]:
for col in ["n_estimators", "max_depth", "learning_rate", "min_child_weight", "subsample", "colsample_bytree", "gamma", "reg_alpha", "reg_lambda"]:
    print(f"=== {col} ===")
    display(summarize_param_effect(xgb_rs, col).round(4))


=== n_estimators ===


,n_estimators,n_trials,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_std_avg,roc_auc_mean_avg,recall_mean_avg,fit_time_mean_avg_s
0,200,24,0.9557,0.9560,0.0085,0.9825,0.9242,0.1331
1,500,22,0.9556,0.9556,0.0089,0.9825,0.9171,0.2101
2,300,29,0.9551,0.9559,0.0089,0.9824,0.9212,0.1769
3,700,25,0.9544,0.9544,0.0093,0.9820,0.9097,0.2881


=== max_depth ===


,max_depth,n_trials,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_std_avg,roc_auc_mean_avg,recall_mean_avg,fit_time_mean_avg_s
0,3,21,0.9583,0.9591,0.0087,0.9836,0.9425,0.1328
1,5,15,0.9569,0.9573,0.0090,0.9830,0.9313,0.2026
2,9,17,0.9552,0.9548,0.0080,0.9823,0.9082,0.2111
3,7,15,0.9550,0.9559,0.0091,0.9823,0.9176,0.2167
4,12,11,0.9532,0.9536,0.0094,0.9816,0.9041,0.2373
5,20,21,0.9519,0.9531,0.0094,0.9812,0.9002,0.2320


=== learning_rate ===


...

,gamma,n_trials,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_std_avg,roc_auc_mean_avg,recall_mean_avg,fit_time_mean_avg_s
0,0.5,30,0.9566,0.9575,0.0089,0.9829,0.9279,0.1828
1,0.3,28,0.9549,0.9549,0.0087,0.9822,0.9146,0.1847
2,0.1,17,0.9546,0.9548,0.0090,0.9822,0.9128,0.2032
3,0.0,25,0.9542,0.9546,0.0091,0.9820,0.9139,0.2416


=== reg_alpha ===


,reg_alpha,n_trials,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_std_avg,roc_auc_mean_avg,recall_mean_avg,fit_time_mean_avg_s
0,0.5,19,0.9563,0.9571,0.0087,0.9827,0.9235,0.2092
1,0.0,30,0.9552,0.9556,0.0089,0.9824,0.9227,0.2086
2,1.0,22,0.9548,0.9549,0.0093,0.9822,0.9136,0.2255
3,0.1,29,0.9547,0.9549,0.0087,0.9822,0.9133,0.1708


=== reg_lambda ===


,reg_lambda,n_trials,pr_auc_mean_avg,pr_auc_mean_median,pr_auc_std_avg,roc_auc_mean_avg,recall_mean_avg,fit_time_mean_avg_s
0,10.0,29,0.9562,0.9560,0.0088,0.9827,0.9234,0.2039
1,2.0,18,0.9557,0.9554,0.0085,0.9825,0.9195,0.1630
2,5.0,23,0.9556,0.9556,0.0089,0.9825,0.9166,0.2371
3,1.0,30,0.9535,0.9547,0.0093,0.9818,0.9134,0.1950


#### Conclusão

- n_estimators
  
    O lift favorecia 700 e 500, mas o ganho marginal mostra que 200 e 500 têm as melhores médias de PR-AUC, com 200 ligeiramente na frente e bem mais barato. 700 piora tanto em desempenho quanto em custo.
    Interpretação: eu não priorizaria 700. A região mais interessante parece ser 200–500, com centro em 200–300 se quiser mais parcimônia, ou incluindo 500 se quiser manter flexibilidade.

- max_depth
  
    Aqui a convergência é muito forte. O lift já mostrava dominância de 3, e o ganho marginal confirma isso com folga: max_depth=3 tem melhor PR-AUC, melhor ROC-AUC, melhor Recall e menor custo.
    Interpretação: esse é o parâmetro mais claro da análise. A região principal é 3–5, com forte preferência por 3.

- learning_rate
  
    O lift favorecia 0.03 e 0.01, e o ganho marginal confirma isso. 0.03 entrega a melhor média de PR-AUC, enquanto 0.01 mantém desempenho muito próximo e melhora Recall. 0.10 cai claramente.
    Interpretação: a melhor faixa está entre 0.01 e 0.03, com 0.03 como centro e 0.01 como alternativa mais conservadora.

- min_child_weight
  
    O lift favorecia 3, e o ganho marginal também coloca 3 levemente na frente em PR-AUC. Os valores 1 e 7 continuam próximos, enquanto 5 perde um pouco.
    Interpretação: eu priorizaria 1–3, com leve preferência por 3.

- subsample
  
    O lift favorecia 1.0, e o ganho marginal confirma isso com a melhor média de PR-AUC, melhor ROC-AUC e melhor Recall. 0.7 e 0.85 ficam próximos, mas abaixo.
    Interpretação: 1.0 deve ser o centro da busca, podendo manter 0.7 como alternativa secundária.

- colsample_bytree
    O lift era quase neutro entre 0.7 e 1.0, e o ganho marginal continua mostrando pouca separação. 1.0 tem melhor PR-AUC, enquanto 0.7 tem Recall um pouco melhor e menor custo.
    Interpretação: faz sentido manter 0.7 e 1.0 no espaço reduzido. 0.85 perde prioridade.

- gamma
  
    O lift já favorecia 0.5, e o ganho marginal confirma com boa margem: gamma=0.5 é o melhor em média e ainda mais barato que 0.0.
    Interpretação: esse é outro sinal forte. A região principal é 0.3–0.5, com clara preferência por 0.5.

- reg_alpha
  
    O lift sugeria 0.0, 0.5 e 1.0, mas o ganho marginal melhora a leitura: 0.5 aparece como melhor valor médio, seguido de 0.0. 1.0 já perde mais.
    Interpretação: o espaço mais promissor parece ser 0.0–0.5, com destaque para 0.5.

- reg_lambda
  
    O lift já apontava 10.0, e o ganho marginal confirma isso: 10.0 tem a melhor média de PR-AUC, enquanto 2.0 e 5.0 seguem próximos. 1.0 cai bastante.
    Interpretação: a melhor faixa está entre 2.0 e 10.0, com preferência por 10.0.

### Grid Paramétrico Reduzido para o Optuna

In [ ]:
xgb_optuna_search_space = {
    "model__n_estimators": [200, 250, 300],
    "model__max_depth": [3, 4, 5],
    "model__learning_rate": {"low": 0.01, "high": 0.03, "log": True},
    "model__min_child_weight": [1, 2, 3],
    "model__subsample": [0.7],
    "model__colsample_bytree": [0.7],
    "model__gamma": [0.3, 0.4, 0.5],
    "model__reg_alpha": {"low": 0.0, "high": 0.5, "step": 0.1},
    "model__reg_lambda": {"low": 2.0, "high": 10.0, "step": 2.0},
}


### Etapa 2 - Otimização Bayesiana (Optuna)